# Description

In this notebook, I will load the Llama3-2 weight and try quantize it

In [1]:
import os
import sys
import time
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import OrderedDict
import re 
import json
import math
import numpy as np
import pandas as pd
from tqdm import tqdm

from datasets import load_dataset

import torch
torch.manual_seed(123)
import torch.nn.functional as F

from utils.model import Llama3Model, generate, text_to_token_ids, token_ids_to_text
from utils.tokenizer import Llama3Tokenizer, ChatFormat, clean_text
from utils.model import LLAMA32_CONFIG_1B, LLAMA32_CONFIG_3B

/home/tnguyen10/.conda/envs/gpu_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Load model

In [2]:
# ===== Hyper-parameter =====
# MODEL_FILE = "/home/tnguyen10/Desktop/llm/model/llama3.2-1B-instruct.pth"
MODEL_FILE = "/scratch/tnguyen10/llama3.2-3B-instruct.pth"
MODEL_CONTEXT_LENGTH = 8192  # Support up to 131_072
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
TOP_K = 1
TOKENIZER_FILE = "/home/tnguyen10/Desktop/llm/model/tokenizer.model"
DEVICE = "cuda"

In [3]:
# 1. Load model
if os.path.exists(MODEL_FILE) == False:
    print(f"[ERROR] Model does not exist !!!")
    sys.exit(0)

if "1B" in MODEL_FILE:
    llama32_config = LLAMA32_CONFIG_1B
elif "3B" in MODEL_FILE:
    llama32_config = LLAMA32_CONFIG_3B
else:
    print(f"[ERROR] Check model file again !!!")
    sys.exit(0)

llama32_config["context_length"] = MODEL_CONTEXT_LENGTH

model = Llama3Model(llama32_config)
checkpoint = torch.load(MODEL_FILE, weights_only=True, map_location=DEVICE)
model.load_state_dict(checkpoint)
model.to(DEVICE)

print(f"Model {MODEL_FILE} loaded successfully !!!")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params/1e6:.2f} M")

Model /scratch/tnguyen10/llama3.2-3B-instruct.pth loaded successfully !!!
Total parameters: 3606.75 M


In [4]:
# 2. Load tokenizer
if not os.path.exists(TOKENIZER_FILE):
    print(f"[ERROR] Does not have TOKENIZER_FILE")
    sys.exit(0)

tokenizer = Llama3Tokenizer(TOKENIZER_FILE)

if "instruct" in MODEL_FILE:
    tokenizer = ChatFormat(tokenizer)
print(f"Tokenizer loaded successfully.")

Tokenizer loaded successfully.


In [5]:
input_prompt = "What is the capital of VietNam?"
print(f"Input prompt: {input_prompt}")
print('-' * 80)

# 3. Generate text
start = time.time()

token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_prompt, tokenizer).to(DEVICE),
    max_new_tokens=MAX_NEW_TOKENS,
    context_size=llama32_config["context_length"],
    top_k=TOP_K,
    temperature=TEMPERATURE,
)

print(f"Generation time: {time.time() - start:.2f} sec")

output_text = token_ids_to_text(token_ids, tokenizer)
output_text = clean_text(output_text)
print("\nOutput text:\n", output_text)

Input prompt: What is the capital of VietNam?
--------------------------------------------------------------------------------



 [WARNING]: Reached limit of 512 token without generating an EOS token. 

Generation time: 12.93 sec

Output text:
 The capital of Vietnam is Hanoi.


# 2. Evaluate Perplexity

In [6]:
def compute_perplexity(model, tokenizer, sentences, device="cuda",\
    max_ctx=2048,          # model context window
    window_stride=None,    # if None, use max_ctx - 1 (full overlap by 1)
    add_bos=True,          # prepend BOS if your model expects it
    batch_size=8,          # micro-batching over windows
    ):
    model.eval()
    total_nll = 0.0
    total_tokens = 0

    # Try to get special ids if available
    bos_id = getattr(tokenizer, "bos_token_id", None)
    eos_id = getattr(tokenizer, "eos_token_id", None)
    pad_id = getattr(tokenizer, "pad_token_id", -100)  # for ignore_index if you pad

    if window_stride is None:
        window_stride = max_ctx - 1  # predict token t from t-1 with maximal coverage

    with torch.no_grad():
        for text in sentences:
            ids = text_to_token_ids(text, tokenizer).squeeze(0)  # [T]
            if add_bos and (bos_id is not None):
                ids = torch.cat([torch.tensor([bos_id], dtype=ids.dtype), ids], dim=0)
            T = ids.numel()
            if T < 2:
                continue

            # Create overlapping windows over the token sequence
            starts = list(range(0, max(1, T-1), window_stride))
            window_inputs = []
            window_targets = []
            for s in starts:
                e = min(T, s + max_ctx)
                x = ids[s:e]                           # [L]
                if x.numel() < 2:
                    continue
                inp = x[:-1]                           # predict next tokens
                tgt = x[1:]
                window_inputs.append(inp)
                window_targets.append(tgt)

            # Mini-batch the windows (pad to equal length; mask pads)
            i = 0
            while i < len(window_inputs):
                batch_inp = window_inputs[i:i+batch_size]
                batch_tgt = window_targets[i:i+batch_size]
                i += batch_size

                max_len = max(x.size(0) for x in batch_inp)
                inp_pad = pad_id if pad_id is not None else -100  # just for consistent shape
                tgt_pad = -100  # ignore_index for loss

                inp_tensor = torch.full((len(batch_inp), max_len), inp_pad, dtype=torch.long)
                tgt_tensor = torch.full((len(batch_tgt), max_len), tgt_pad, dtype=torch.long)
                for r, (inp, tgt) in enumerate(zip(batch_inp, batch_tgt)):
                    L = inp.size(0)
                    inp_tensor[r, :L] = inp
                    tgt_tensor[r, :L] = tgt

                inp_tensor = inp_tensor.to(device)
                tgt_tensor = tgt_tensor.to(device)

                # Forward pass: model should ignore pad_id naturally if it embeds it;
                # we only compute loss on positions where tgt != -100.
                logits = model(inp_tensor)             # [B, L, V]
                log_probs = F.log_softmax(logits, dim=-1)
                # gather log p(correct)
                t = tgt_tensor.unsqueeze(-1)           # [B, L, 1]
                token_logp = log_probs.gather(-1, t).squeeze(-1)  # [B, L]

                # mask out padding targets (-100)
                mask = (tgt_tensor != -100)
                nll = -(token_logp * mask).sum().item()
                count = mask.sum().item()

                total_nll += nll
                total_tokens += count

    ppl = math.exp(total_nll / max(1, total_tokens))
    return ppl

In [7]:
# ----- Load 1,000 samples from WikiText2 -----
def load_wikitext2_samples(n=1000, min_length=10):
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
    # Filter out empty or too-short lines
    samples = [x["text"] for x in dataset if len(x["text"].strip()) > min_length]
    return samples[:n]

In [8]:
num_samples = 10_000

samples = load_wikitext2_samples(num_samples)
print(f"Loaded {len(samples)} samples. Computing perplexity...")

ppl = compute_perplexity(model, tokenizer, samples, DEVICE)
print(f"\nPerplexity on {len(samples)} WikiText2 samples: {ppl:.2f}")

Loaded 2454 samples. Computing perplexity...

Perplexity on 2454 WikiText2 samples: 44.34
